# 04 — Evaluation and Predictions

This notebook loads the trained models and evaluates them against real 2025 market data:
- **Predictions** — run each saved model on unseen 2025 prices
- **Accuracy metrics** — MAE, RMSE, MAPE, and directional accuracy
- **Plots** — predicted vs actual closing price for each ETF
- **Portfolio summary** — side-by-side comparison across all four ETFs

## 1. Imports

In [ ]:
import sys
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
sys.path.append(os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

from data_loader import TICKERS, TEST_START, TEST_END
from model import FEATURES, SEQUENCE_LENGTH, MODEL_DIR, AttentionLayer
from evaluate import (
    load_saved_model, prepare_test_data,
    make_predictions, calculate_metrics, plot_predictions
)

print('All imports successful')
print(f'Test window : {TEST_START} → {TEST_END}')
print(f'ETFs        : {TICKERS}')

## 2. Verify Saved Models Exist

In [ ]:
all_ready = True

for ticker in TICKERS:
    model_path  = f'{MODEL_DIR}/{ticker}_model.keras'
    scaler_path = f'{MODEL_DIR}/{ticker}_scaler.pkl'
    model_ok    = os.path.exists(model_path)
    scaler_ok   = os.path.exists(scaler_path)
    status      = 'ready' if (model_ok and scaler_ok) else 'MISSING'
    print(f'{ticker}:  model {"✓" if model_ok else "✗"}  scaler {"✓" if scaler_ok else "✗"}  → {status}')
    if not (model_ok and scaler_ok):
        all_ready = False

if not all_ready:
    print('\nOne or more models are missing — run notebook 03 first.')
else:
    print('\nAll models present — ready to evaluate.')

## 3. Run Predictions — All ETFs

Loads each saved model, downloads 2025 test data, engineers the same features as training, and generates predictions.

In [ ]:
results = {}

for ticker in TICKERS:
    print(f"\n{'='*50}")
    print(f'  {ticker}')
    print(f"{'='*50}")

    model, scaler = load_saved_model(ticker)
    X_test, y_test, df_test = prepare_test_data(ticker, model, scaler)
    predictions = make_predictions(model, scaler, X_test)

    # Inverse transform actual values
    close_idx = FEATURES.index('Close')
    dummy_actual = np.zeros((len(y_test), len(FEATURES)))
    dummy_actual[:, close_idx] = y_test
    actual = scaler.inverse_transform(dummy_actual)[:, close_idx]

    results[ticker] = {
        'predictions': predictions,
        'actual'     : actual,
        'df_test'    : df_test
    }

    print(f'  Sequences evaluated : {len(X_test)}')
    print(f'  Predicted range     : ${predictions.min():.2f} – ${predictions.max():.2f}')
    print(f'  Actual range        : ${actual.min():.2f} – ${actual.max():.2f}')

print(f"\n{'='*50}")
print('All predictions complete.')

## 4. Accuracy Metrics — All ETFs

In [ ]:
def calculate_directional_accuracy(actual, predicted):
    actual_dir    = np.diff(actual) > 0
    predicted_dir = np.diff(predicted) > 0
    correct = np.sum(actual_dir == predicted_dir)
    total   = len(actual_dir)
    return (correct / total) * 100

metrics_rows = []

for ticker in TICKERS:
    actual      = results[ticker]['actual']
    predictions = results[ticker]['predictions']

    mae     = np.mean(np.abs(actual - predictions))
    rmse    = np.sqrt(np.mean((actual - predictions) ** 2))
    mape    = np.mean(np.abs((actual - predictions) / actual)) * 100
    dir_acc = calculate_directional_accuracy(actual, predictions)

    results[ticker]['mae']     = mae
    results[ticker]['rmse']    = rmse
    results[ticker]['mape']    = mape
    results[ticker]['dir_acc'] = dir_acc

    metrics_rows.append({
        'Ticker'             : ticker,
        'MAE ($)'            : f'${mae:.2f}',
        'RMSE ($)'           : f'${rmse:.2f}',
        'MAPE (%)'           : f'{mape:.2f}%',
        'Directional Acc (%)': f'{dir_acc:.2f}%'
    })

metrics_df = pd.DataFrame(metrics_rows).set_index('Ticker')
display(metrics_df)

## 5. Predicted vs Actual — Individual Charts

One chart per ETF showing the full 2025 prediction vs reality.

In [ ]:
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for i, ticker in enumerate(TICKERS):
    actual      = results[ticker]['actual']
    predictions = results[ticker]['predictions']
    df_test     = results[ticker]['df_test']
    dates       = df_test.index[SEQUENCE_LENGTH:]

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(dates, actual,      label='Actual price',    color=colors[i], linewidth=1.5)
    ax.plot(dates, predictions, label='Predicted price', color=colors[i], linewidth=1.5,
            linestyle='--', alpha=0.8)
    ax.fill_between(dates, actual, predictions, alpha=0.08, color=colors[i])

    mape    = results[ticker]['mape']
    dir_acc = results[ticker]['dir_acc']
    ax.set_title(f'{ticker} — Predicted vs Actual Closing Price (2025)\n'
                 f'MAPE: {mape:.2f}%   |   Directional Accuracy: {dir_acc:.2f}%', fontsize=11)
    ax.set_ylabel('Price (AUD)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    os.makedirs('plots', exist_ok=True)
    save_path = f'plots/{ticker}_prediction.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {save_path}\n')

## 6. All ETFs — Side by Side

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, ticker in enumerate(TICKERS):
    actual      = results[ticker]['actual']
    predictions = results[ticker]['predictions']
    dates       = results[ticker]['df_test'].index[SEQUENCE_LENGTH:]
    mape        = results[ticker]['mape']
    dir_acc     = results[ticker]['dir_acc']

    axes[i].plot(dates, actual,      color=colors[i], linewidth=1.3, label='Actual')
    axes[i].plot(dates, predictions, color=colors[i], linewidth=1.3, label='Predicted',
                 linestyle='--', alpha=0.8)
    axes[i].fill_between(dates, actual, predictions, alpha=0.07, color=colors[i])
    axes[i].set_title(f'{ticker}\nMAPE {mape:.2f}%  |  Dir acc {dir_acc:.2f}%', fontsize=10)
    axes[i].set_ylabel('Price (AUD)')
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    axes[i].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(axes[i].xaxis.get_majorticklabels(), rotation=30, ha='right')
    axes[i].legend(fontsize=8)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Predicted vs Actual Closing Prices — All ETFs (2025)', fontsize=13)
plt.tight_layout()
plt.savefig('plots/portfolio_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → plots/portfolio_summary.png')

## 7. Prediction Error Distribution

Shows how prediction errors are distributed — a well-calibrated model has errors centred near zero.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
axes = axes.flatten()

for i, ticker in enumerate(TICKERS):
    errors = results[ticker]['actual'] - results[ticker]['predictions']
    axes[i].hist(errors, bins=40, color=colors[i], alpha=0.75, edgecolor='white')
    axes[i].axvline(0,            color='black', linestyle='--', linewidth=1)
    axes[i].axvline(errors.mean(), color='red',  linestyle='--', linewidth=1,
                    label=f'Mean error: ${errors.mean():.2f}')
    axes[i].set_title(f'{ticker} — Prediction Error Distribution')
    axes[i].set_xlabel('Error (Actual − Predicted) AUD')
    axes[i].set_ylabel('Frequency')
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Prediction Error Distributions — 2025 Test Data', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Directional Accuracy — Monthly Breakdown

Breaks directional accuracy down by month to see if the model performs better at certain times of year.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, ticker in enumerate(TICKERS):
    actual      = results[ticker]['actual']
    predictions = results[ticker]['predictions']
    dates       = results[ticker]['df_test'].index[SEQUENCE_LENGTH:]

    actual_dir    = np.diff(actual) > 0
    predicted_dir = np.diff(predictions) > 0
    correct       = (actual_dir == predicted_dir).astype(int)

    acc_df = pd.DataFrame({'correct': correct}, index=dates[1:])
    monthly = acc_df.resample('ME').mean() * 100

    bar_colors = ['#4CAF50' if v >= 50 else '#E53935' for v in monthly['correct']]
    axes[i].bar(monthly.index, monthly['correct'], color=bar_colors, alpha=0.8, width=20)
    axes[i].axhline(50, color='black', linestyle='--', linewidth=1, label='50% baseline')
    axes[i].set_title(f'{ticker} — Monthly Directional Accuracy')
    axes[i].set_ylabel('Accuracy (%)')
    axes[i].set_ylim(0, 100)
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.3, axis='y')

plt.suptitle('Monthly Directional Accuracy — 2025 (green = above 50%)', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Portfolio Summary

In [ ]:
print(f"{'='*58}")
print(f"{'PORTFOLIO EVALUATION SUMMARY — 2025':^58}")
print(f"{'='*58}")
print(f"{'Ticker':<10} {'MAE':>8} {'RMSE':>8} {'MAPE':>8} {'Dir Acc':>10}")
print(f"{'-'*58}")

for ticker in TICKERS:
    r = results[ticker]
    print(f"{ticker:<10} ${r['mae']:>6.2f} ${r['rmse']:>6.2f} {r['mape']:>6.2f}%  {r['dir_acc']:>7.2f}%")

print(f"{'-'*58}")
avg_mape    = np.mean([results[t]['mape']    for t in TICKERS])
avg_dir_acc = np.mean([results[t]['dir_acc'] for t in TICKERS])
print(f"{'Portfolio avg':<10} {'':>8} {'':>8} {avg_mape:>6.2f}%  {avg_dir_acc:>7.2f}%")
print(f"{'='*58}")
print(f'\nKey: MAPE = mean absolute % error | Dir Acc = % of days correct direction predicted')
print(f'Directional accuracy > 50% means the model beats random guessing.')

## Summary

| | |
|---|---|
| Test window | January 2025 – present |
| Metrics | MAE, RMSE, MAPE, Directional Accuracy |
| Plots saved | `plots/{ticker}_prediction.png`, `plots/portfolio_summary.png` |
| Best performer | See MAPE column above |
| Weakest performer | See MAPE column above |

Results and observations are documented in the [README](README.md).